In [2]:
from neural_network import normalize, train_neural_network, DATA_HEADERS
import pandas as pd
import numpy as np

C:\Users\ecorb\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
url = "https://raw.githubusercontent.com/reganq/csc311-project/refs/heads/main/cleaned_data.csv"
df = pd.read_csv(url)

In [4]:
# split the data
train_df = df[df['is_train'] == True]
val_df = df[df['is_train'] == False]

# split off the labels - they cannot be one-hot encoded for logreg. Instead, we have 3 targets (0, 1, 2)
t_train = np.argmax(np.stack([
    (train_df['painting'] == 'The Persistence of Memory').astype(np.int8),
    (train_df['painting'] == 'The Starry Night').astype(np.int8),
    (train_df['painting'] == 'The Water Lily Pond').astype(np.int8)
]), axis=0)

t_val = np.argmax(np.stack([
    (val_df['painting'] == 'The Persistence of Memory').astype(np.int8),
    (val_df['painting'] == 'The Starry Night').astype(np.int8),
    (val_df['painting'] == 'The Water Lily Pond').astype(np.int8)
]), axis=0)

headers = DATA_HEADERS

X_train = np.array(train_df.get(headers))
X_val = np.array(val_df.get(headers))

In [5]:
# normalize the data
X_train = normalize(X_train)
X_val = normalize(X_val)

In [6]:
def build_all_models(alpha,
                     activation,
                     batch_size,
                     hidden_layer_sizes,
                     X_train=X_train,
                     t_train=t_train,
                     X_valid=X_val,
                     t_valid=t_val):
    """
    Returns a dictionary, `out`, whose keys are the the hyperparameter choices, and whose values are
    the training and validation accuracies (via the `score()` method).
    Arguments:
        - alpha: Regularization strength.
        - activation: Which activation function to use.
        - batch_size: Size of the mini-batches for stochastic optimizers.
        - hidden_layer_sizes: Tuple of integers. The i-th element represents the number of neurons in the i-th hidden layer.
    """
    out = {}

    for a in alpha:
        for act in activation:
            for b in batch_size:
                for h in hidden_layer_sizes:
                    out[(a, act, b, h)] = {}
                    # Create a neural network model based on the given hyperparameters and fit it to the data
                    model = train_neural_network(X_train, t_train, alpha=a, activation=act, batch_size=b, hidden_layer_sizes=h)
                    
                    # store the validation and training scores in the `out` dictionary
                    out[(a, act, b, h)]['val'] = model.score(X_valid, t_valid)
                    out[(a, act, b, h)]['train'] = model.score(X_train, t_train)
    return out

In [8]:
# Hyperparameters values
alpha = [0.0001, 0.001, 0.01]
activation = ['relu', 'tanh']
batch_size = [16, 32]
hidden_layer_sizes = [(50,), (100,), (50, 50)]

res = build_all_models(alpha=alpha, activation=activation, batch_size=batch_size, hidden_layer_sizes=hidden_layer_sizes) # call `build_all_models` for the given hyperparameters

# search for the optimal (max_depth, min_samples_split) given this criterion
max_score = 0
best_params = None
for a, act, b, h in res:
    if res[(a, act, b, h)]['val'] > max_score:
        max_score = res[(a, act, b, h)]['val']
        best_params = (a, act, b, h)

print(f"Best parameters: {best_params}")
print(f"Best score: {max_score}")

C:\Users\ecorb\AppData\Roaming\Python\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\ecorb\AppData\Roaming\Python\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\ecorb\AppData\Roaming\Python\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\ecorb\AppData\Roaming\Python\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.wa

Best parameters: (0.0001, 'tanh', 32, (100,))
Best score: 0.9056047197640118
